# Bloque 5 · Del script al sistema — prácticas en Google Colab
## De un programa que funciona a un sistema que se mantiene

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/consumer-spain/curso-ia-colab/blob/main/bloque5/bloque5_practicas.ipynb)

**Esto es el Plan B.** Si en tu ordenador algo no funciona (Python, paquetes, permisos…), puedes seguir aquí
**las prácticas del Bloque 5**, en el mismo orden que en clase.

**Cómo se usa**
- Cada bloque gris es una *celda*. Se ejecuta con el botón ▶ de su izquierda (o **Mayús + Enter**).
- Ejecuta las celdas **en orden**, de arriba abajo.
- Si Colab se desconecta o lo reinicias, vuelve a ejecutar la celda **Preparación**. Si la repites, no pasa nada: no duplica nada.
- Colab **no guarda tus cambios**. Si quieres conservarlos: *Archivo → Guardar una copia en Drive*.
- No subas aquí datos reales ni información interna: todo el material del curso es de prueba.

| En clase | Aquí |
|---|---|
| Doble clic en `COMPROBAR_BLOQUE5.bat` / `PREPARAR_ENTORNO_BLOQUE5.bat` | Ejecutar la celda **Preparación** |
| DB Browser for SQLite, para abrir la base de ejemplo | Celdas con `sqlite3` y pandas (`ver_tabla`, `ejecutar_sql`) |
| Programador de tareas de Windows | **No exportable** (Colab no es Windows). Una celda programa una tarea equivalente dentro de la sesión |
| Excel, el Explorador de archivos | pandas (`ver_excel`, `ver_csv`) y `ver_imagenes` |
| El programa del reto, en tu equipo | `%%writefile` + `!python`, como en el Bloque 1 |
| Healthchecks.io, Telegram, API de una IA | **Igual**, en tu navegador o en tu móvil. No se hacen desde el cuaderno |

> **Importante.** Lo que se obtiene desde Colab sale de servidores de Google, no de un equipo institucional:
> **no tiene valor probatorio**. La IP pública de salida es la de un centro de datos de Google y la hora es la del reloj de la
> máquina virtual de Colab. Es una réplica de las prácticas para aprender, no una forma de producir evidencia.

## Preparación

Descarga los materiales del Bloque 5 (el mismo ZIP que en clase), instala lo que falte (httpx, openpyxl, pillow, Playwright y el
navegador Chromium) y ejecuta el comprobador. Es el equivalente a `PREPARAR_ENTORNO_BLOQUE5.bat` + `COMPROBAR_BLOQUE5.bat`.
Debe terminar en **TODO LISTO**.

*Adaptada.* La primera vez instala Chromium y tarda **cerca de un minuto**; la celda va contando los pasos.
El comprobador hace **una** petición de lectura a la web de Mercadona, que sale desde Google.

In [ ]:
# PREPARACIÓN · ejecuta esta celda primero
import os, sys, time, zipfile, subprocess, urllib.request
from pathlib import Path

ZIP_URL = "https://raw.githubusercontent.com/consumer-spain/curso-ia-colab/main/bloque5/BLOQUE5_ALUMNADO_FINAL.zip"
TRABAJO = Path.cwd().resolve()
if "BLOQUE5_ALUMNADO" in TRABAJO.parts:      # si se vuelve a ejecutar esta celda, no se anida otra carpeta
    TRABAJO = Path(*TRABAJO.parts[:TRABAJO.parts.index("BLOQUE5_ALUMNADO")])
BASE = TRABAJO / "BLOQUE5_ALUMNADO"
os.chdir(TRABAJO)
t0 = time.time()

print("[1/4] Materiales del Bloque 5")
if not BASE.exists():
    urllib.request.urlretrieve(ZIP_URL, TRABAJO / "materiales_b5.zip")
    with zipfile.ZipFile(TRABAJO / "materiales_b5.zip") as z:
        z.extractall(TRABAJO)
    print("      descargados en", BASE)
else:
    print("      ya estaban descargados:", BASE)

print("[2/4] Paquetes de Python")
for modulo, paquete in (("pandas", "pandas"), ("openpyxl", "openpyxl"), ("httpx", "httpx"),
                        ("PIL", "pillow"), ("playwright", "playwright")):
    try:
        __import__(modulo)
        print("      ya instalado:", paquete)
    except ImportError:
        print("      instalando", paquete, "...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)


def chromium_listo():
    r = subprocess.run([sys.executable, "-c",
        "from pathlib import Path\nfrom playwright.sync_api import sync_playwright\n"
        "p = sync_playwright().start()\nprint(Path(p.chromium.executable_path).exists())\np.stop()"],
        capture_output=True, text=True)
    return r.stdout.strip() == "True"


print("[3/4] Navegador Chromium de Playwright")
if chromium_listo():
    print("      ya instalado")
else:
    print("      instalando (la primera vez tarda cerca de un minuto; no cierres la pestaña)...")
    r = subprocess.run([sys.executable, "-m", "playwright", "install", "--with-deps", "chromium"],
                       capture_output=True, text=True)
    if r.returncode != 0 or not chromium_listo():
        print("      NO se ha podido instalar. Última salida:")
        print("\n".join((r.stdout + r.stderr).splitlines()[-8:]))
    else:
        print(f"      instalado ({time.time() - t0:.0f} s en total)")

import pandas as pd
pd.set_option("display.max_columns", None, "display.width", 250, "display.max_colwidth", 70)
from IPython.display import Image as VerImagen, Markdown, display


def ir_a(carpeta=""):
    """Se coloca en una carpeta del material (como abrir la carpeta en el Explorador). La crea si no existe."""
    destino = BASE / carpeta if carpeta else BASE
    destino.mkdir(parents=True, exist_ok=True)
    os.chdir(destino)
    print("Estás en:", Path.cwd().relative_to(TRABAJO))


def ver_excel(ruta, hoja=None):
    """Muestra un Excel (en clase se abre con Excel): lista las hojas y enseña la que pidas (por defecto, la primera)."""
    xl = pd.ExcelFile(ruta)
    print("Hojas:", " · ".join(xl.sheet_names))
    return xl.parse(xl.sheet_names[0] if hoja is None else hoja)


def ver_csv(ruta, filas=15):
    """Muestra las primeras filas de un CSV."""
    return pd.read_csv(ruta, encoding="utf-8-sig").head(filas)


def ver_imagenes(carpeta=".", n=3, ancho=420):
    """Muestra las primeras imágenes de una carpeta (en clase: abrirlas en el Explorador)."""
    archivos = sorted(p for p in Path(carpeta).rglob("*") if p.suffix.lower() in (".png", ".jpg", ".jpeg"))
    if not archivos:
        print("No hay imágenes en", carpeta)
    for p in archivos[:n]:
        print(p)
        display(VerImagen(filename=str(p), width=ancho))
    if len(archivos) > n:
        print(f"(hay {len(archivos)} imágenes; se muestran {n})")


def mostrar(archivo):
    """Muestra un documento del material (Markdown) ya formateado."""
    display(Markdown((BASE / archivo).read_text(encoding="utf-8")))


def descargar(*rutas):
    """Descarga archivos a tu ordenador."""
    try:
        from google.colab import files
        for r in rutas:
            files.download(str(r))
    except ImportError:
        print("Esta celda solo descarga archivos dentro de Google Colab.")


print("[4/4] Comprobador del Bloque 5")
ir_a("")
r = subprocess.run([sys.executable, "COMPROBAR_BLOQUE5.py"], capture_output=True, text=True)
print(r.stdout.rstrip())
if r.returncode != 0:
    print("\nHAY ALGO QUE REVISAR: copia el mensaje anterior y pídele ayuda al docente.")

---
# A · Apertura

## Lo que ya sabemos hacer
- **Bloque 3 · OBTENER:** sacar datos de una web.
- **Bloque 4 · CONSERVAR:** guardar lo que vimos y vincularlo a la observación (captura, URL, IP, fecha y hora, código postal).

## Hoy: INTEGRAR en un único programa
Un programa que **obtiene y conserva** en la misma ejecución, con **comprobaciones** a lo largo de todo el proceso.
Después, cuatro módulos **opcionales** (no son una cadena): **HISTÓRICO · PROGRAMACIÓN · SUPERVISIÓN · ADAPTAR**.

### El encargo nuevo
*Vigilar precios durante meses.* ¿Qué necesita un programa para no ser solo «un script que funcionó una vez»?
Piénsalo antes de seguir: ¿qué guardarías, quién lo ejecutaría mañana y cómo te enterarías de que ha fallado?

---
# B · HISTÓRICO

**Una observación aislada no permite comparar.** Para ver cómo cambia un precio hay que **guardar cada observación con su
contexto** (qué producto, en qué ejecución, cuándo, dónde, a qué precio) y poder consultarlas después.

> **IGUAL** (en clase se proyectan datos reales del Geoportal: Ajalvir, el identificador `IDEESS`, las estaciones que aparecen y
> desaparecen entre dos fechas). Aquí trabajamos con una **base de ejemplo con datos ficticios**, la misma del material de clase.

## ¿Cabe en un Excel?
Cada día se guardan las observaciones de todas las estaciones. Prueba con cifras distintas (el formulario cambia con los números;
la hoja de Excel admite 1.048.576 filas). *Supuesto: el mismo número de estaciones cada día; en la realidad varía.*

In [ ]:
#@title ¿Cabe en una hoja de Excel? { display-mode: "form" }
ESTACIONES = 11521 #@param {type:"integer"}
DIAS = 91 #@param {type:"integer"}

LIMITE = 1_048_576          # filas de una hoja de Excel
filas = ESTACIONES * DIAS
puntos = lambda n: f"{n:,}".replace(",", ".")
print(f"{puntos(ESTACIONES)} estaciones × {DIAS} días = {puntos(filas)} filas")
print("Límite de una hoja de Excel:", puntos(LIMITE), "filas")
print("-> CABE" if filas <= LIMITE else "-> NO CABE")
print("Días que caben con esas estaciones:", LIMITE // ESTACIONES)

## Una base de datos relacional
Un conjunto de **tablas relacionadas entre sí por códigos comunes**. Para el histórico, tres: **productos**, **ejecuciones** y
**observaciones**. En clase se abre `BLOQUE5_EJEMPLO_RELACIONAL.sqlite` con DB Browser for SQLite; aquí, con celdas.
*(Es un archivo de datos ficticios; solo se abre para **leer**: ninguna consulta puede modificarlo.)*

In [ ]:
import sqlite3
DB = BASE / "BLOQUE5_EJEMPLO_RELACIONAL.sqlite"


def consulta(sql):
    """Ejecuta una consulta SELECT sobre la base de ejemplo (solo lectura) y devuelve una tabla."""
    con = sqlite3.connect(DB.as_uri() + "?mode=ro", uri=True)
    try:
        return pd.read_sql_query(sql, con)
    finally:
        con.close()


def ver_tabla(nombre):
    """Equivale a la pestaña «Browse Data» de DB Browser."""
    if nombre not in consulta("SELECT name FROM sqlite_master WHERE type='table'")["name"].tolist():
        raise ValueError(f"No existe la tabla {nombre!r}")
    return consulta(f'SELECT * FROM "{nombre}"')


print("Tablas de la base:", consulta("SELECT name FROM sqlite_master WHERE type='table'")["name"].tolist())
ver_tabla("productos")

### Mira las otras dos tablas
Fíjate en los **identificadores compartidos**: `producto_id` y `ejecucion_id` son los que **conectan** las tablas.
Y en `archivo_captura`: `NULL` significa *esa observación no tiene captura* (solo se captura una muestra).

In [ ]:
ver_tabla("ejecuciones")

In [ ]:
ver_tabla("observaciones")

## La IA puede proponer el diseño
No hace falta saber diseñar la base de antemano: primero se aclara el encargo y después se propone la solución.
Las tres tablas anteriores son **un ejemplo**, no una estructura válida para todos los proyectos.

**Prompt (cópialo en tu IA):**
> «Voy a recoger precios periódicamente. Quiero conservar productos, ejecuciones, precios, URL, IP y capturas de una muestra.
>
> Antes de diseñar la base de datos, hazme preguntas de una en una sobre los usos, las comparaciones y los controles que necesitaré.
> Después propón una estructura sencilla y explícame cómo comprobarla.»

## La pregunta al histórico
**¿Qué productos subieron más entre el 21 y el 22 de octubre?** Mismos productos, misma zona.

### Le pedimos la consulta a la IA, con una frase
**Prompt (cópialo en tu IA):**
> «Con las tablas productos, ejecuciones y observaciones, compara el 21 con el 22 de octubre para los mismos productos y la misma zona,
> calcula las variaciones y ordena las subidas.»

La IA devuelve una **consulta** (SQL): un texto técnico que **no hay que memorizar ni leer línea a línea**. Lo importante es pedirla,
ejecutarla y **revisar el resultado**: ¿mismos productos? ¿misma zona? ¿cuadra con las tablas que has visto?

*Adaptada.* En clase se pega en DB Browser. Aquí: **pega la consulta de tu IA debajo de la primera línea** de la celda siguiente
(deja `%%writefile` arriba) y ejecútala; después, la celda que la lanza. Si te falla, copia el **error completo** y vuelve a la IA, como en el Bloque 1.

In [ ]:
ir_a("HISTORICO")

In [ ]:
%%writefile mi_consulta.sql
-- Pega aquí, debajo de esta línea, la consulta que te ha dado la IA

In [ ]:
ir_a("HISTORICO")


def ejecutar_sql(archivo):
    """Ejecuta la consulta de un archivo .sql sobre la base de ejemplo y muestra el resultado."""
    texto = Path(archivo).read_text(encoding="utf-8")
    if not [l for l in texto.splitlines() if l.strip() and not l.strip().startswith("--")]:
        print(f"{archivo} todavía no tiene ninguna consulta: pega la de tu IA debajo de la primera línea.")
        return None
    try:
        return consulta(texto)
    except Exception as e:
        print("La consulta ha fallado:", e)
        print("Copia este mensaje completo y la consulta a tu IA.")
        return None


ejecutar_sql("mi_consulta.sql")

<details><summary>Si no tienes la IA a mano: ejecutar la consulta del material</summary>

El ZIP del material incluye `CONSULTA_BBDD_DEMO.sql`, la consulta que se proyecta en clase. Ejecútala y compara con la tuya.
</details>

In [ ]:
ir_a("")
ejecutar_sql("CONSULTA_BBDD_DEMO.sql")

### ¿Lo daríais por bueno?
Mira el resultado y vuelve a las tablas: ¿los **mismos productos** en las dos ejecuciones? ¿la **misma zona** (código postal)?
¿El cambio coincide con los precios de `observaciones`? **Una consulta que se ejecuta no es una consulta correcta.**

---
# C · PROGRAMACIÓN

**Ya tenemos histórico. ¿Quién ejecuta el programa mañana?** Para que se ejecute sin que estés delante hace falta
**una tarea programada**, que necesita tres cosas: **cuándo** (desencadenante), **qué** (acción) y **dónde** (carpeta de trabajo).

> **No exportable.** En clase se usa el **Programador de tareas de Windows** (*Crear tarea básica → nombre → desencadenante → acción →
> finalizar*). Colab no es Windows y no puede programar tareas del sistema ni mantenerlas cuando se cierra la sesión.
> **Equivalente aproximado:** la celda siguiente programa una tarea sencilla **dentro de esta sesión** (una vez, dentro de unos minutos)
> y te deja comprobar, como en la biblioteca del Programador, si se ha ejecutado. La idea es la misma; la herramienta, no.

## Práctica · programa una tarea sencilla
En clase: *nombre `CURSO_IA_DEMO` · una vez, dentro de unos minutos · acción: iniciar `notepad.exe` · finalizar y comprobar si se abre.*
Aquí, la acción es escribir un archivo (en Colab no hay Bloc de notas que abrir). Elige un tiempo corto para esperar poco.

In [ ]:
#@title Programar una tarea sencilla { display-mode: "form" }
import threading, datetime as dt
NOMBRE = "CURSO_IA_DEMO" #@param {type:"string"}
MINUTOS = 1 #@param {type:"number"}

try:
    TAREAS                                   # si repites la celda, se conservan las tareas anteriores
except NameError:
    TAREAS = {}


def programar_tarea(nombre, minutos):
    viejo = TAREAS.get(nombre)
    if viejo and viejo["estado"] == "programada":
        viejo["cancelar"].set()              # sustituir una tarea con el mismo nombre
    ahora = dt.datetime.now().astimezone()
    t = {"nombre": nombre, "cuando": ahora + dt.timedelta(minutes=minutos), "estado": "programada", "ultima": None,
         "cancelar": threading.Event(), "archivo": TRABAJO / f"{nombre}_resultado.txt"}

    def lanzar():
        if t["cancelar"].wait(timeout=minutos * 60):
            t["estado"] = "sustituida"
            return
        t["archivo"].write_text(f"Tarea {nombre} ejecutada a las {dt.datetime.now().astimezone():%H:%M:%S %Z}\n", encoding="utf-8")
        t["ultima"] = dt.datetime.now().astimezone()
        t["estado"] = "ejecutada"

    threading.Thread(target=lanzar, daemon=True).start()
    TAREAS[nombre] = t
    print(f"Tarea {nombre!r} creada. Se ejecutará una vez, a las {t['cuando']:%H:%M:%S %Z} (hora de este equipo).")


def ver_tareas():
    """Equivale a la biblioteca del Programador de tareas: ¿está la tarea? ¿hay una última ejecución?"""
    if not TAREAS:
        print("No hay ninguna tarea programada.")
    for t in TAREAS.values():
        ult = f"{t['ultima']:%H:%M:%S %Z}" if t["ultima"] else "—"
        print(f"{t['nombre']:<16} estado: {t['estado']:<10} programada para: {t['cuando']:%H:%M:%S %Z}   última ejecución: {ult}")


programar_tarea(NOMBRE, MINUTOS)

### ¿Se ejecutó la tarea que acabas de crear?
Espera a la hora indicada y ejecuta la celda siguiente (puedes repetirla). Observa: ¿está la tarea en la biblioteca?
¿Aparece una **última ejecución**? ¿Se creó el archivo?

**¿Demuestra eso que un scraper habría recogido datos correctos?** Vemos que *algo se lanzó*. No vemos si el programa obtuvo productos
ni capturas correctas.

In [ ]:
ver_tareas()
print()
archivo = TAREAS[NOMBRE]["archivo"] if "NOMBRE" in globals() and NOMBRE in TAREAS else None
if archivo is not None and archivo.exists():
    print("Archivo creado por la tarea:", archivo.name)
    print(archivo.read_text(encoding="utf-8"))
else:
    print("Todavía no hay archivo (la tarea no se ha ejecutado aún).")

### Para Python en Windows: un archivo `.bat`
En clase se prueba con `notepad.exe`, un programa que Windows sabe iniciar. Para lanzar Python lo más controlable es **indicar el
intérprete o usar un `.bat`** que fija la carpeta de trabajo, llama a Python y guarda lo que escribe. *(No se ejecuta aquí: es de Windows.)*

```bat
@echo off
cd /d "%~dp0"
if not exist logs mkdir logs
py -3 mi_programa.py >> logs\lanzador.log 2>&1
```

---
# D · SUPERVISIÓN

**La tarea se ejecutó, pero ¿terminó bien el trabajo?** Dos herramientas complementarias:
- **El log** reconstruye lo ocurrido (pero hay que abrirlo, y si el programa no llegó a ejecutarse, no existe).
- **Una señal externa** (Healthchecks.io) avisa cuando la señal esperada **no llega**, aunque nadie mire.

> **IGUAL / no exportable.** Aquí no hay nada que ejecutar: Healthchecks.io se usa **en tu navegador** (con tu propia cuenta) y el
> log lo genera el programa. La URL privada de un check **no se escribe en una celda ni se comparte** con una IA ni en un documento.

## Un log cuenta qué ocurrió en cada ejecución
Extracto de un log (de una **prueba sintética local**, no de Mercadona real):

```
22:46:26  INICIO Bloque 5 · CP 28004
22:46:26  INFO   Categorías completadas: 6/6 · productos únicos: 12
22:46:31  INFO   CP 28004 confirmado en la interfaz de la web.
22:46:33  INFO   Capturas válidas: 3 de 3
[…]
22:46:33  CTRL   CAPTURAS         OK
22:46:33  CTRL   CONTEXTO_CP      OK
22:46:33  CTRL   CONTEXTO_IP      OK
22:46:33  CTRL   CONTRASTE_MANUAL PENDIENTE
[…]
22:46:33  FIN    Estado: OK_AUTOMATICO
```
Tres preguntas: **¿qué hizo?** (pasos y recuentos) · **¿qué comprobó?** (controles y su estado) · **¿cómo terminó?** (estado final).

**Prompt para pedir un log útil (cópialo en tu IA):**
> «En cada ejecución crea un archivo en la carpeta logs con fecha y hora, fuente, zona, recuentos por fase, errores y estado final.
> No sobrescribas ejecuciones anteriores. No guardes contraseñas ni tokens. Si hay un fallo, dime cómo copiar el log para que podamos diagnosticarlo.»

## Si nadie lee el log, necesitamos una señal externa
En el panel de Healthchecks.io se crea un *check* con **Period** (cada cuánto esperamos la señal) y **Grace Time** (margen de tolerancia).
Si la señal no llega a tiempo, avisa (**DOWN**); cuando vuelve, avisa otra vez (**UP**). Una señal recibida **no certifica que los datos sean correctos**.

**Prompt para pedir la señal (cópialo en tu IA):**
> «Quiero añadir un healthcheck opcional. Guardaré su URL privada en HEALTHCHECK_PING_URL. Al iniciar, envía la señal de inicio si procede.
> Envía éxito solo si los controles mínimos se superan; si falla, envía una señal de fallo. Registra también el resultado en el log.»

*Nunca incluyas la URL privada en un documento compartido.* En Colab, si algún día la usas, guárdala en el panel **🔑 Secretos**, no en una celda.

---
# E · ADAPTAR

**¿Qué necesitamos que haga ahora el programa?** De 1.500 productos, ¿qué merece nuestra atención? **Detectar con reglas** (por ejemplo,
una subida superior a un umbral) y **después decidir qué avisar**.

> **IGUAL.** Telegram se usa en tu móvil y el análisis con la API de una IA en la web de su proveedor: no se ejecutan desde este cuaderno.
> Las claves y los *tokens* **nunca** van en una celda ni en un prompt (en Colab, el panel 🔑 Secretos).

**Prompt · avisar de los cambios de precio:**
> «Si una comparación detecta una subida de precio superior al umbral que definamos, quiero recibir una alerta en Telegram con producto,
> precio anterior, precio actual, porcentaje, fecha, URL y enlace a los archivos disponibles. El bot y el chat se configurarán con variables de entorno.»

**Prompt · avisar si algo sale mal** (controles: datos, campos, cobertura, caída anómala):
> «Añade controles al final de cada barrido; distingue ERROR y REVISAR. Guarda un log y envía un aviso a Telegram con un resumen y el archivo si no contiene información sensible.»

## Textos que hay que interpretar
Buscar encuentra datos; **interpretar exige contexto**. Dos anuncios sin número visible: ¿el mismo diagnóstico? Un **indicio no equivale a una infracción acreditada**:
la IA ordena los casos y la valoración jurídica contrasta hechos y normativa vigente.

**Prompt · interpretar con criterios y conservar las dudas** (anuncios ficticios):
> «Analiza el anuncio con los criterios normativos que adjunto.
>
> Cita literalmente los fragmentos relevantes.
>
> Señala lo que no sabes.
>
> Devuelve un resultado estructurado, sin declarar infracciones.»

Estados posibles de cada fila de la tabla de resultados: **REVISAR · SIN INDICIOS · información insuficiente**. Una fila por anuncio, siempre las mismas columnas
y siempre con el identificador del anuncio de origen.

---
# F · RETO FINAL · Mercadona

**Un único programa que obtenga datos de la tienda y conserve una muestra documentada de lo que vio.** Unos 28 minutos de trabajo y 5 de validación.
Un resultado parcial bien explicado es un buen resultado.

*Adaptada.* En clase el programa se ejecuta en tu Windows. Aquí lo escribes con tu IA, lo pegas en una celda `%%writefile` y lo lanzas con `!python`
(en el Bloque 1 lo hacías igual). Playwright y Chromium ya están instalados por la celda **Preparación**.

## El encargo

In [ ]:
mostrar("ENUNCIADO_RETO_MERCADONA.md")

## Punto de partida para la IA
Copia el texto siguiente en tu IA y completa lo que está entre [corchetes]. **Dos cambios para Colab** (hazlos en el texto antes de enviarlo):
- Donde dice *«un programa de Python para Windows»*, escribe *«para Google Colab (Linux, Python 3.13, sin ventanas)»*.
- Al contestar *«cómo ejecuto Python»*: *«el programa se llamará `mi_programa.py`, lo lanzaré con `!python mi_programa.py`, Playwright y Chromium están instalados y deben ejecutarse sin ventana (headless); que muestre por pantalla el avance»*.

In [ ]:
print((BASE / "PROMPT_INICIAL.md").read_text(encoding="utf-8"))

## Tu programa
**1.** Colócate en la carpeta de trabajo del reto.

In [ ]:
ir_a("RETO")

**2.** Pega el código que te dé la IA **debajo de la primera línea** de la celda siguiente (deja `%%writefile mi_programa.py` arriba).
Debe ser el archivo completo.

In [ ]:
%%writefile mi_programa.py
# Pega aquí, debajo de esta línea, el código que te ha dado la IA

**3.** Ejecútalo. **Empieza por una prueba pequeña** (2–3 categorías) y amplía después: una ejecución completa implica muchas peticiones.
Si el programa te pide datos (por ejemplo, el código postal), aparecerá un cuadro de texto. Si falla, copia el **error completo** y el código a la IA,
como en el Bloque 1, y corrige solo esa fase.

In [ ]:
ir_a("RETO")
!python mi_programa.py

## Mira lo que ha producido
En clase se abren el Excel y la carpeta de capturas. Aquí: la lista de archivos, las hojas y las primeras filas, y las capturas.
Cambia las rutas por las de **tu** programa.

In [ ]:
ir_a("RETO")
!find . -maxdepth 3 -not -name "*.pyc" -not -path "./__pycache__*" | sort | head -60

In [ ]:
# Cambia la ruta por la de tu Excel (en este ejemplo no existe: la celda te lo dice)
RUTA_EXCEL = "mi_resultado.xlsx" #@param {type:"string"}
ir_a("RETO")
if Path(RUTA_EXCEL).exists():
    display(ver_excel(RUTA_EXCEL))
else:
    print("No existe", RUTA_EXCEL, "-> cambia el nombre por el de tu Excel (míralo en la lista de archivos de arriba).")

In [ ]:
# Cambia la carpeta por la de tus capturas
CARPETA_CAPTURAS = "." #@param {type:"string"}
ir_a("RETO")
ver_imagenes(CARPETA_CAPTURAS, n=3)

## ¿Y si Mercadona te frena?
**No intentes saltarte** CAPTCHA, bloqueos (401/403/429) ni otras medidas de protección. Si la web te frena, **para**, documenta el límite y avisa al docente.
Esta celda hace **una** petición de lectura para ver si la web responde desde Colab (no verifica tu programa):

In [ ]:
import urllib.request, urllib.error
try:
    with urllib.request.urlopen(urllib.request.Request("https://tienda.mercadona.es/", headers={"User-Agent": "CursoIA-comprobador"}), timeout=10) as r:
        print("La web de Mercadona responde: HTTP", r.status)
except urllib.error.HTTPError as e:
    print("La web ha contestado con HTTP", e.code, "-> no insistas ni busques otra vía; avisa al docente.")
except Exception as e:
    print("No se ha podido abrir la web desde Colab:", type(e).__name__)

## Antes de darlo por bueno · validación
**Contrasta a mano una muestra con la web actual**: nombre, precio y formato de 2–3 productos del Excel frente a la tienda en tu navegador.
Un resultado parcial bien explicado vale más que un total que no puedes demostrar.

In [ ]:
mostrar("CHECKLIST_VALIDACION.md")

### Lo que has obtenido desde Colab: ¿qué valor tiene?
Los datos y las capturas salen de una **máquina virtual de Google**. Mira qué contexto habría registrado tu programa:

In [ ]:
import urllib.request, datetime as dt
try:
    ip = urllib.request.urlopen("https://api.ipify.org", timeout=8).read().decode()
    print("IP pública de salida de este cuaderno:", ip, "(es una IP de centro de datos de Google, no la de tu equipo)")
except Exception:
    print("No se ha podido consultar la IP pública desde este cuaderno.")
print("Hora de este equipo:", dt.datetime.now().astimezone().strftime("%Y-%m-%d %H:%M:%S %Z"), "(reloj de la máquina virtual; no es un sello de tiempo)")

**La IP identifica el canal de salida, no a una persona; la hora es la del equipo, no un sello de tiempo.** Y desde Colab, ni siquiera es un equipo institucional:
**estas capturas y estos datos no tienen valor probatorio**. Sirven para aprender el método.

---
# Cinco bloques. Un mismo método.
**B1 construir · B2 consolidar · B3 obtener · B4 conservar · B5 integrar.** Método común: OBJETIVO → DIAGNÓSTICO → INFORMACIÓN PARA LA IA → SOLUCIÓN → COMPROBACIÓN.

---
# Solo si te has bloqueado

**Restaurar los materiales originales** (por ejemplo, si has estropeado la base de ejemplo o un documento).
Vuelve a extraer el ZIP **sobre** `BLOQUE5_ALUMNADO`: recupera los archivos originales y **no borra** lo que has creado (`mi_programa.py`, la carpeta `RETO`, tus consultas…).

In [ ]:
import urllib.request, zipfile
from pathlib import Path
zip_local = TRABAJO / "materiales_b5.zip"
if not zip_local.exists():
    urllib.request.urlretrieve(ZIP_URL, zip_local)
with zipfile.ZipFile(zip_local) as z:
    z.extractall(TRABAJO)
print("Materiales originales restaurados.")
ir_a("")